# 28 · Cost per million tokens and build vs buy
Level L5 · Part 4 · core · ~25 min · builds on lessons 26, 27

Time: about 25 min. No key, no GPU and no Docker. The arithmetic runs anywhere. Two lab numbers come from the **recorded** runs of lessons 18 and 27 (`data/recorded/`), and they are labelled as such. Prices are public third-party snapshots with their source and date (RESEARCH.md §1, 2026-10-21); this lesson does not claim a vendor price it could not source.

Machine: any laptop with the topic's Python. Ports: none.

## Goal
By the end you can turn a GPU's hourly price and its served tokens into cost per million tokens at 20, 50 and 80 percent utilisation, see that utilisation is the biggest lever, size what a cache hit rate adds (at most prefill share × hit rate), and find the breakevens between on-demand, reserved, spot, on-prem and an API.

## Why it matters
- **Without it:** a team compares the GPU's hourly price with an API's price per token. It forgets that an idle GPU is still paid for. The same on-demand GPU costs about **5x more per token at 20% utilisation than at 100%** (Step 3 prints both). It also ignores the prompt prefixes that a cache could serve, which can cut the work per request.
- **With it:** one formula gives the cost per served token at any utilisation. The same formula shows the cache lever and every breakeven, so a buy decision becomes a number you can check.
- **When you don't need it:** a few thousand requests a month on a hosted API. There the price per token is the whole answer. Do this arithmetic when you run your own GPUs or sign a fixed commitment.

## The idea
A GPU costs money by the hour, busy or idle. So cost per token is the hourly price divided by the tokens served in that hour, and that depends on **utilisation**, the share of capacity in use. Halve the utilisation and you double the cost per token. A **cache hit rate** is a second lever: a prefix already in the KV cache (lesson 14) skips prefill work, so the same GPU serves more requests. It matters most when prefill takes a large share of GPU time. Build versus buy compares that cost with an API's price. The commitment (on-demand, reserved, spot, on-prem) sets the hourly price and risk. Analogy: a taxi paid by the hour, so the cost per ride depends on how many rides you drive.

## Picture
```mermaid
flowchart LR
  P[GPU price per hour] --> D[Cost per million tokens]
  U[Utilisation: share of capacity in use] --> T[Served tokens per hour]
  T --> D
  H[Cache hit rate] -->|less prefill work| T
  D --> B{Build or buy?}
  A[API price per million tokens] --> B
```

## Step by step

### Step 1: the inputs, with their sources
Three inputs. The **price** is one L4 GPU (`g2-standard-4`, us-central1), on-demand: about $0.707 per hour, from a third-party snapshot (RESEARCH.md §1, "GCP GPU prices", 2026-10-21). The official page was not reachable: **check the current price** in the Cloud Billing Catalog. Spot prices come from the same snapshot, whose sources disagree by about 2x, so both ends are kept.

The **throughput** is a teaching value: a ceiling of 1,000 served tokens per second (input plus output). **Utilisation** is served tokens per second divided by that ceiling, so cost is linear in it. It is a cost-model scale, not the queueing utilisation (ρ) of lesson 26. Lab runs of lessons 18 and 27 are not used for cost.

In [1]:
import json, statistics
from pathlib import Path
import pandas as pd

HOURS = 730                                       # hours in a month: 24 x 365 / 12
PRICE_OD = 0.707                                  # $/h, one L4 GPU, on-demand (RESEARCH.md §1, 2026-10-21, third-party: verify)
PRICE_SPOT = {"spot, low": 0.28, "spot, high": 0.62}   # $/h, same snapshot; the sources disagree by 2x
TPS = 1000                                        # served tokens/s at 100% utilisation: a TEACHING VALUE, not a benchmark
print(f"inputs: on-demand ${PRICE_OD}/h, spot ${PRICE_SPOT['spot, low']}-{PRICE_SPOT['spot, high']}/h, {TPS:,} served tokens/s at 100%")

inputs: on-demand $0.707/h, spot $0.28-0.62/h, 1,000 served tokens/s at 100%


### Step 2: the traffic mix, from lesson 07
The 30 requests of lesson 07's support trace set the mix: about 1,550 input tokens and 69 output tokens per request. The trace is made-up support traffic from a fixed-seed generator, not a real log, so it has no licence to check. Here is the first row, then the means.

In [2]:
rows = [json.loads(l) for l in Path("../data/07/support_trace.jsonl").read_text().splitlines()]
print("first row:", rows[0])
IN = statistics.mean(r["prompt_tokens"] for r in rows)
OUT = statistics.mean(r["output_tokens"] for r in rows)
print(f"lesson 07 trace: {len(rows)} requests, mean {IN:,.0f} input + {OUT:.0f} output tokens per request")

first row: {'prompt_tokens': 1560, 'output_tokens': 24}
lesson 07 trace: 30 requests, mean 1,553 input + 69 output tokens per request


### Step 3: illustrative cost per million served tokens at 20, 50, 80 and 100%
The table is built with pandas: one column per commitment, one row per utilisation. Each cell is the hourly price divided by the tokens served in that hour, times a million. It is **illustrative**: on-demand and spot are third-party values to check, and the throughput is a teaching value. The formula is written out by hand in **Under the hood**, after the table.

In [3]:
prices = {"on-demand": PRICE_OD, **PRICE_SPOT}
utils = [0.2, 0.5, 0.8, 1.0]
cost = pd.DataFrame({name: [p / (TPS * 3600 * u) * 1e6 for u in utils] for name, p in prices.items()},
                    index=[f"{u:.0%}" for u in utils])
cost.index.name = "utilisation"
print("illustrative $ per million served tokens\n", cost.round(3).to_string())
print(f"on-demand at 20% costs {cost.loc['20%', 'on-demand'] / cost.loc['100%', 'on-demand']:.1f}x the cost at 100%")
print(f"one support request ({IN + OUT:,.0f} tokens) at 50% on-demand: ${(IN + OUT) * cost.loc['50%', 'on-demand'] / 1e6:.5f}; 1,000 requests: ${(IN + OUT) * 1000 * cost.loc['50%', 'on-demand'] / 1e6:.2f}")

illustrative $ per million served tokens
              on-demand  spot, low  spot, high
utilisation                                  
20%              0.982      0.389       0.861
50%              0.393      0.156       0.344
80%              0.245      0.097       0.215
100%             0.196      0.078       0.172
on-demand at 20% costs 5.0x the cost at 100%
one support request (1,622 tokens) at 50% on-demand: $0.00064; 1,000 requests: $0.64


**Read it.** Cost falls as utilisation rises, for every commitment alike. Lesson 26 targets about 70% utilisation, and its table shows a p95 wait of 0 at 80%, about 330 ms at 90% and about 1 s at 95%. So plan at 70 to 80%. Above 80% the saving is small (0.245 $/M at 80% against 0.196 at 100%) and the tail grows.

### Step 4: how much a cache hit rate lowers the cost
A prefix cache serves the shared part of a prompt from memory, so the GPU skips that prefill. **Derivation.** Let `s` be the prefill share of GPU time with no cache, and `h` the hit rate on input tokens. The time left is `(1 − s) + s × (1 − h)`, so the saving is `s × h`. Example: `s = 0.5` and `h = 0.45` give `0.5 × 0.45 = 22.5%`, and the cost is 77.5% of the uncached cost.

Get `s` from one request with no queue: prefill share is about TTFT divided by the total time (lesson 07 measures both). The hit rates are sources, and the lab row is the lab's own hit rate. Each is from RESEARCH.md §4 (Meta, Cloudflare) and §8 (DeepSeek).

In [4]:
rec18 = json.loads(Path("../data/recorded/18.json").read_text())
A, B = rec18["A"], rec18["B"]
print(f"lesson 18 CPU lab (recorded): first token p50 {A['p50']:.1f} s without prefix cache, {B['p50']:.1f} s with it (hit rate {B['hit_rate']:.1%})")
base50 = cost.loc["50%", "on-demand"]
hits = {"no prefix cache": 0.0, "Meta, 40-50% (midpoint)": 0.45, "DeepSeek, 24 h": 0.563,
        "Cloudflare, with affinity": 0.80, "lesson 18 lab (upper bound)": B["hit_rate"]}
shares = {"5%": 0.05, "20%": 0.20, "50%": 0.50}      # prefill share of GPU time: scenarios, not measured on a GPU
table = pd.DataFrame(index=list(hits))
for k, s in shares.items():
    table[f"$/M, prefill {k}"] = [base50 * (1 - s * h) for h in hits.values()]
    table[f"saved %, prefill {k}"] = [100 * s * h for h in hits.values()]
print("on-demand at 50% utilisation\n", table.round(3).to_string())

lesson 18 CPU lab (recorded): first token p50 21.2 s without prefix cache, 1.1 s with it (hit rate 96.9%)
on-demand at 50% utilisation
                              $/M, prefill 5%  saved %, prefill 5%  $/M, prefill 20%  saved %, prefill 20%  $/M, prefill 50%  saved %, prefill 50%
no prefix cache                        0.393                0.000             0.393                  0.00             0.393                  0.00
Meta, 40-50% (midpoint)                0.384                2.250             0.357                  9.00             0.304                 22.50
DeepSeek, 24 h                         0.382                2.815             0.349                 11.26             0.282                 28.15
Cloudflare, with affinity              0.377                4.000             0.330                 16.00             0.236                 40.00
lesson 18 lab (upper bound)            0.374                4.845             0.317                 19.38             0.202           

**Read it.** The saving is at most `s × h`. At 5% prefill the lab's 97% hit saves about 5%. At 50% prefill the saving runs from 23% (Meta's midpoint) to 49% (the lab row). The lab row is an upper bound: a 97% hit on one shared prompt, with a prefill share we did not measure on a GPU. Measure both on your own trace before you count on the lever.

### Step 5: breakeven, reserved vs on-demand (worked)
On-demand is paid only for the hours the GPU runs. Reserved is paid for every hour, at a lower rate. So reserved wins once the GPU runs for more than the rate ratio of the month. No reserved rate is sourced here: the 70% ratio is an **example**, to replace with your quote.

In [5]:
reserved_h = 0.70 * PRICE_OD                       # $/h, example: reserved at 70% of on-demand
reserved_month = reserved_h * HOURS
print(f"reserved ${reserved_h:.3f}/h x {HOURS} h = ${reserved_month:.0f} a month, whatever the GPU does")
for share in (0.40, 0.80):                         # share of the month the GPU runs
    od_month = PRICE_OD * HOURS * share
    print(f"runs {share:.0%} of the month: on-demand ${od_month:.0f} -> {'on-demand' if od_month < reserved_month else 'reserved'} wins")
print(f"breakeven share = reserved / on-demand = {0.70:.0%}")

reserved $0.495/h x 730 h = $361 a month, whatever the GPU does
runs 40% of the month: on-demand $206 -> on-demand wins
runs 80% of the month: on-demand $413 -> reserved wins
breakeven share = reserved / on-demand = 70%


### Step 6: breakeven, on-prem and spot
On-prem is paid every hour too, so it wins above the same kind of ratio: its all-in $/h over on-demand. The on-prem rates are **placeholders** for your own hardware, power and hosting cost. Spot wins per served token while the work it loses stays under the breakeven: `1 − spot ÷ on-demand`, from the sourced spot range.

In [6]:
for p in (0.40, 0.60, 0.80):                       # placeholder all-in on-prem $/h
    share = p / PRICE_OD
    verdict = f"wins above {share:.0%} of the month" if share < 1 else "never: on-demand is cheaper at every share of hours"
    print(f"on-prem at ${p:.2f}/h: {verdict}")
for name, s in PRICE_SPOT.items():
    print(f"{name} (${s}/h): spot wins if interruptions waste less than {1 - s / PRICE_OD:.0%} of the work")

on-prem at $0.40/h: wins above 57% of the month
on-prem at $0.60/h: wins above 85% of the month
on-prem at $0.80/h: never: on-demand is cheaper at every share of hours
spot, low ($0.28/h): spot wins if interruptions waste less than 60% of the work
spot, high ($0.62/h): spot wins if interruptions waste less than 12% of the work


### Step 7: build vs buy against an API
APIs charge a different rate for input and output tokens, so compare with a **blended** price: each rate weighted by its share of the tokens. Choose the shares from your own traffic. The lesson 07 mix is about 96% input tokens, so the blend is close to the input rate. The API prices below are example grid values; copy the real ones from your provider's page. The breakeven compares the API with the 100% utilisation cost, the best case for self-hosting.

In [7]:
IN_SHARE = IN / (IN + OUT)
print(f"blend weights from the trace: {IN_SHARE:.1%} input tokens, {1 - IN_SHARE:.1%} output tokens")
cost100 = cost.loc["100%", "on-demand"]
for api in (0.25, 0.50, 1.00):                     # example blended $ per million tokens
    u = cost100 / api
    verdict = f"self-host wins above {u:.0%} utilisation" if u <= 1 else "never: the API is cheaper at any utilisation"
    print(f"blended API at ${api:.2f} per million tokens: {verdict}")

blend weights from the trace: 95.7% input tokens, 4.3% output tokens
blended API at $0.25 per million tokens: self-host wins above 79% utilisation
blended API at $0.50 per million tokens: self-host wins above 39% utilisation
blended API at $1.00 per million tokens: self-host wins above 20% utilisation


## Under the hood
The pandas table is a library version of one formula. Here is the same calculation by hand, with the units spelled out, and a check that it agrees with the table. The formula is the whole lesson: price per hour over tokens per hour, with a cache factor.

In [8]:
def cost_per_million(price_per_hour, utilisation, served_tps=TPS, hit=0.0, prefill_share=0.0):
    """Dollars per million served tokens: one hour's price over the tokens served in that hour."""
    tokens_per_hour = served_tps * 3600 * utilisation        # capacity in one hour x the share in use
    gpu_time_factor = 1 - prefill_share * hit                # a cache hit skips the prefill work it covers
    return price_per_hour / tokens_per_hour * 1e6 * gpu_time_factor

assert abs(cost_per_million(PRICE_OD, 0.5) - cost.loc["50%", "on-demand"]) < 1e-12
assert abs(cost_per_million(PRICE_OD, 0.5, hit=0.563, prefill_share=0.5) - table.loc["DeepSeek, 24 h", "$/M, prefill 50%"]) < 1e-12
print("hand-built formula matches the tables:", round(cost_per_million(PRICE_OD, 0.2), 3), "at 20%")

hand-built formula matches the tables: 0.982 at 20%


## At work
- **DeepSeek** reported its own 24-hour figures (Feb 2025): $87,072 a day at $2 per GPU-hour on about 227 nodes of 8 GPUs. DeepSeek reports a 56.3% hit on 608B input tokens (RESEARCH.md §8), and Mooncake builds its whole design around a pooled KV cache. Whether that saving is large depends on your own prefill share (Step 4). → lesson 31 (scale down when load falls).
- **Cloudflare** (2026-05) moved its peak input cache hit ratio from 60% to 80% with a session-affinity header, so routing sets the cache lever. → lesson 30 (prefix-aware routing).
- **Meta and PyTorch** (2025-09): sticky routing and load balancing gave a 40 to 50% prefix cache hit at 90% HBM use. → lesson 30.
- **Databricks** says to benchmark your own deployment: a vendor's tokens per second depends on hardware and lengths. → lesson 27.
- Go deeper: quantisation (lessons 20 and 21) lowers cost per token after a quality check.

## Common mistakes
**1. Dividing by utilisation zero.** An idle GPU has a price but serves no tokens, so cost per token is undefined. The cell below does that division. **This cell fails on purpose.**

In [9]:
PRICE_OD / (TPS * 3600 * 0.0) * 1e6              # an idle hour: no tokens served; this cell fails on purpose

ZeroDivisionError: float division by zero

**The fix:** report the idle cost per hour (the bill) next to the cost per served token, and count the idle hours as cost. Never average a cost per token over hours when the GPU was idle.

## Production checklist
- Compare options per **served token at the utilisation you plan for** (70 to 80%), not per GPU-hour.
- Use the served tokens per second from your own GPU run (lesson 27's method, spread lengths, an SLO), not a vendor's figure.
- Measure the hit rate from the server's counters (`vllm:prefix_cache_hits_total` over `vllm:prefix_cache_queries_total`, RESEARCH.md §7a) and the prefill share from TTFT, before you count the lever.
- Price spot with its retry path and the work it wastes. Spot is cheaper only if the lost share stays under the breakeven.
- Record the source and the date of every price in the notebook, and check it again before a decision.

## Try it
**Paying for wasted work.** If 15% of the work is lost, you pay for 100 units to keep 85. So the price per kept unit is the price divided by the share kept. Worked once: $0.28 with 15% lost is $0.28 ÷ 0.85 = $0.329 per hour of kept work.

In [10]:
eff = 0.28 / (1 - 0.15)                            # the pattern: price / share kept
print(f"$0.28/h with 15% lost: ${eff:.3f} per hour of kept work")

$0.28/h with 15% lost: $0.329 per hour of kept work


Now do the same for a spot GPU at $0.62 per hour that loses 30% of its work. Is its cost per million served tokens at 50% utilisation below on-demand's? Compare with the on-demand value in Step 3.

In [11]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [12]:
eff_high = 0.62 / (1 - 0.30)
spot_cost = cost_per_million(eff_high, 0.5)
od_cost = cost_per_million(PRICE_OD, 0.5)
print(f"effective spot: ${eff_high:.3f}/h of kept work; at 50%: {spot_cost:.3f} $/M vs on-demand {od_cost:.3f} $/M")
print("cheaper per served token:", spot_cost < od_cost)

effective spot: $0.886/h of kept work; at 50%: 0.492 $/M vs on-demand 0.393 $/M
cheaper per served token: False


## Recap
- Utilisation is the bigger lever: cost per served token is about 5x higher at 20% than at 100% on-demand, and it is linear in utilisation.
- A cache hit rate saves at most `prefill share × hit rate`. It is a major lever only when prefill is a large share of GPU time, so measure that share on your trace.
- Reserved and on-prem win above their rate ratio of the month. Spot wins while lost work stays under its breakeven.
- Compare with an API using a blended price, weighted by your own input and output tokens, and plan at 70 to 80% utilisation.
- **Interview:** "When is self-hosting an LLM cheaper than an API?" Strong answer: compare cost per served token at the utilisation you can hold (70 to 80%), with the cache hit rate and the commitment type, against the API's blended price per million tokens; self-host only when your traffic keeps the GPUs busy.